# JCAS simulator

A configuration-driven joint communication-and-sensing (JCAS) network simulator: base stations, mobile UEs, and sensing objects on a toroidal (periodic) window, RT/Rayleigh-fading channel models, Kalman/Extended Kalman filtering, optional sector beamforming and TDD scheduling, and a separate simplified non-captive tracking model.

This notebook exercises the full `SimulationConfig` API directly. For a point-and-click version with the same plots, see `simulator_interface.py` (`streamlit run simulator_interface.py`) — see the repo's `README.md` for setup.

**Association convention:** all association metrics (`association_ratio` and Pearson) are computed on the original **linear values**. Logarithmic axes, when automatically selected, affect visualization only.

## Setup

Install dependencies once (see the repo's `setup.sh` for a scripted version that also creates a virtual environment):

```bash
pip install -r requirements.txt
```

In [ ]:
from jcas_simulator import (
    JCASSimulator, SimulationConfig, NetworkConfig, RegionConfig,
    PointProcessConfig, PopulationConfig, MotionConfig, ChannelConfig,
    FilterConfig, ObservationConfig, NonCaptiveToyModelConfig,
    BeamformingConfig, TDDConfig, TDDPhaseConfig,
)
import numpy as np
from jcas_simulator.visualization import (
    plot_voronoi_network, plot_sinr_kde, plot_covariance_trace_kde,
    plot_workload_kde, plot_entity_trajectories,
    plot_interference_association_scatter,
    plot_sinr_association_scatter,
    plot_filter_queue_association_scatter,
    plot_non_captive_toy_model_estimation_comparison,
)

## Large-scale simulator network example

Every field not set explicitly below uses its default from `jcas_simulator/config.py` — the RT channel model, captive Gauss-Markov mobility, and uniform placement are all defaults. This run uses the ray-traced channel and takes roughly 20-30 seconds.

In [ ]:
config = SimulationConfig(
    master_seed=42,
    horizon=200,
    network=NetworkConfig(
        region=RegionConfig(width=3000.0, height=3000.0),
        base_stations=PointProcessConfig(fixed_count=30),
        ue_population=PopulationConfig(count_model="fixed", fixed_per_cell=1),
        so_population=PopulationConfig(count_model="fixed", fixed_per_cell=1),
    ),
    # Faster-moving entities than the 1.0 m/s-scale default process noise.
    ue_motion=MotionConfig(process_noise_std=1.5),
    so_motion=MotionConfig(process_noise_std=1.5),
    # Remove thermal noise so SINR is interference-limited.
    channel=ChannelConfig(noise_psd_dbm_per_hz=-np.inf),
    filtering=FilterConfig(observation=ObservationConfig(kind="linear")),
)

result = JCASSimulator(config).run()
result.summary()

### Visualization

The plotting layer only consumes the stored `result`; it never reruns network, mobility, channel, queue, or filtering computations. Every function below also accepts `save_path=...` to write the figure to disk.

In [ ]:
plot_voronoi_network(result, show=True)
plot_sinr_kde(result, steady_state=0.4, show=True)
plot_covariance_trace_kde(result, steady_state=0.4, show=True)
plot_workload_kde(result, steady_state=0.4, show=True)

#### Communication–sensing association and trajectories

Association is computed on linear values; log axes below are display-only.

In [ ]:
plot_interference_association_scatter(result, show=True, xscale="log", yscale="log")
plot_sinr_association_scatter(result, show=True, xscale="log", yscale="log")
plot_filter_queue_association_scatter(result, show=True)
plot_entity_trajectories(result, show=True)

## Channel-law ablation: RT vs Rayleigh-fading

This uses the same large-scale simulator and changes only the configured physical channel law. Kept at a smaller scale so the ray-traced run stays fast (a few seconds).

In [ ]:
from dataclasses import replace

exp_config = SimulationConfig(
    master_seed=42,
    horizon=150,
    network=NetworkConfig(
        region=RegionConfig(width=3000.0, height=3000.0),
        base_stations=PointProcessConfig(fixed_count=20),
        ue_population=PopulationConfig(count_model="fixed", fixed_per_cell=1),
        so_population=PopulationConfig(count_model="fixed", fixed_per_cell=1),
    ),
    channel=ChannelConfig(model="exponential", noise_psd_dbm_per_hz=-np.inf),
    filtering=FilterConfig(observation=ObservationConfig(kind="linear")),
)
rt_config = replace(exp_config, channel=replace(exp_config.channel, model="rt"))

exp_result = JCASSimulator(exp_config).run()
rt_result = JCASSimulator(rt_config).run()

ratios = lambda r: {key: values["association_ratio"] for key, values in r.association.items()}
print("Exponential association ratio:", ratios(exp_result))
print("RT association ratio:", ratios(rt_result))

## Non-captive supplied model
Non-captive mode is a separate simulation strategy because the supplied non-captive model has different state/measurement equations.

In [ ]:
non_captive_toy_model_config = SimulationConfig(
    master_seed=42,
    operation_mode="non_captive_toy_model",
    non_captive_toy_model=NonCaptiveToyModelConfig(horizon=200, smoothing_window=20),
)
non_captive_toy_model_result = JCASSimulator(non_captive_toy_model_config).run()
non_captive_toy_model_result.mode, non_captive_toy_model_result.true_state.shape

### Non-captive estimation comparison
This figure is defined only for the non-captive result and uses its stored JCAS error, sensing-only error, and relative state.

In [ ]:
plot_non_captive_toy_model_estimation_comparison(non_captive_toy_model_result, show=True)

### Non-captive distributions
The supplied non-captive strategy has no 2-D Voronoi network or Lindley workload, but its retained SNR and covariance outputs use the same metric-plot API.

In [ ]:
plot_sinr_kde(non_captive_toy_model_result, steady_state=0.4, show=True)
plot_covariance_trace_kde(non_captive_toy_model_result, steady_state=0.4, show=True)

## Nonlinear EKF
Switch to a nonlinear observation by changing only the filtering configuration.

In [ ]:
ekf_config = replace(
    config,
    filtering=FilterConfig(kind="ekf", observation=ObservationConfig(kind="range_bearing")),
)
ekf_result = JCASSimulator(ekf_config).run()
ekf_result.summary()

## Optional sector beamforming and logical scheduling
These are opt-in for the large-scale simulator. The default active scheduling profile remains joint communication+sensing; the example below demonstrates an explicit user-configured communication/sensing cycle. Beamforming requires the exponential channel model, so this reuses `exp_config` from the ablation above.

In [ ]:
beam_tdd_config = replace(
    exp_config,
    beamforming=BeamformingConfig(enabled=True),
    tdd=TDDConfig(enabled=True, phases=(
        TDDPhaseConfig("communication", 2, True, False),
        TDDPhaseConfig("sensing", 1, False, True),
    )),
)
beam_tdd_result = JCASSimulator(beam_tdd_config).run()
beam_tdd_result.tdd_phase_names, beam_tdd_result.beam_indices[:3]